# Example: Stress-Testing a Minimum-Variance Portfolio
In this example, we construct an analytical minimum-variance portfolio and evaluate how return, correlation, and transaction-cost shocks affect its forward performance.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct a minimum-variance allocation:__ Compute the fully invested analytical solution from a covariance matrix.
> * __Design forward stress scenarios:__ Modify expected returns, correlations, and trading costs while holding the evaluation protocol fixed.
> * __Interpret a tail-risk scorecard:__ Compare terminal wealth, drawdown, failure probability, and turnover across scenarios.

Let's test whether an optimized allocation remains convincing away from its calibration assumptions.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Construct the Baseline Portfolio
We create six synthetic assets with annualized expected growth rates and volatilities. For a fully invested portfolio with no additional constraints, the global minimum-variance weights are
$$
\mathbf w_{\min}=
\frac{\boldsymbol\Sigma^{-1}\mathbf 1}
{\mathbf 1^{\mathsf T}\boldsymbol\Sigma^{-1}\mathbf 1}.
$$
The calculation is deterministic and supplies the common target allocation used in every stress scenario.


In [ ]:
Random.seed!(5660);

tickers = ["ALFA", "BRAV", "CHAR", "DELT", "ECHO", "FOXT"];
μannual = [0.075, 0.090, 0.105, 0.070, 0.115, 0.085];
σannual = [0.16, 0.20, 0.24, 0.14, 0.28, 0.18];
ρbase = 0.25;

Σannual = covariance_from_volatility(σannual, ρbase);
wtarget = minimum_variance_weights(Σannual);

allocation_df = DataFrame(ticker=tickers, target_weight=wtarget,
    annual_growth=μannual, annual_volatility=σannual);
pretty_table(allocation_df; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 2: Define the Rebalancing Simulation
Each path begins with one unit of wealth. Asset weights drift with realized returns and are reset to the target every 21 trading days. Rebalancing incurs a proportional cost based on one-way turnover,
$$
\tau_t=\frac{1}{2}\sum_i\left|w_{i,t}^{\text{target}}-w_{i,t}^{\text{pretrade}}\right|.
$$
The simulator records terminal wealth, maximum drawdown, and cumulative turnover.


In [ ]:
# `simulate_rebalanced_path` and `evaluate_rebalancing_scenario` are package methods.


## Task 3: Build and Compare the Stress Scorecard
We compare four cases:

1. the calibration assumptions;
2. a broad reduction in expected returns;
3. a correlation spike;
4. the correlation spike combined with higher transaction costs.

The same target allocation and random seed are used throughout, so the comparison isolates the scenario assumptions.


In [ ]:
scenarios = [
    (name="Baseline", μ=μannual, ρ=ρbase, cost=0.001),
    (name="Return shock", μ=μannual .- 0.08, ρ=ρbase, cost=0.001),
    (name="Correlation spike", μ=μannual, ρ=0.75, cost=0.001),
    (name="Correlation + cost", μ=μannual .- 0.04, ρ=0.75, cost=0.010),
];

scorecard = DataFrame();
for (index, scenario) in enumerate(scenarios)
    Σscenario = covariance_from_volatility(σannual, scenario.ρ);
    result = evaluate_rebalancing_scenario(5660, scenario.μ, Σscenario, wtarget;
        cost_rate=scenario.cost);
    push!(scorecard, merge((scenario=scenario.name,), result); cols=:union);
end

pretty_table(scorecard; table_format=TextTableFormat(borders=text_table_borders__simple))


### Interpretation
An optimizer answers a conditional question: which allocation is best under a declared input model? The scorecard asks a different question: how fragile is that answer when the inputs and implementation conditions change?

Use the failure probability and fifth-percentile wealth to judge downside exposure, drawdown to judge path-dependent pain, and turnover to judge implementation burden. No single metric is sufficient.


## Summary
This example evaluated an optimized portfolio as a deployed process rather than as a single vector of weights.

> __Key Takeaways:__
>
> * __Optimization and validation answer different questions:__ The minimum-variance solution is conditional on its covariance estimate, while stress testing measures sensitivity to alternative conditions.
> * __Dependence shocks matter:__ Diversification can weaken precisely when correlations rise.
> * __Implementation belongs in the model:__ Rebalancing frequency, turnover, and transaction costs change realized performance.

The stress scorecard provides evidence for the trigger and risk-limit design used in the rebalancing lecture.
___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. The synthetic scenarios are simplified and do not represent the full range of market or implementation risks.
